## Action Responder Prompt

This notebook hopes to:

- Gather trace data for action-responder agent
- Structure and save as mlflow dataset
- Utilize judges to evaluate current agent
- Try with better model to confirm judges are working
- Run GEPA optimzation to improve

In [37]:
import mlflow
import pandas as pd
from mlflow.tracking import MlflowClient

mlflow.tracing.enable_notebook_display()

client = MlflowClient()
experiment = mlflow.get_experiment_by_name("summit-sim")


# Get all traces from last 7 days (adjust as needed)
traces = client.search_traces(
    locations=[experiment.experiment_id],
    filter_string="tags.graph_type = 'simulation' AND tags.agent_name = 'action-responder'",
    max_results=500,
)

In [40]:
spans = [
    span
    for trace in traces
    for span in client.get_trace(trace.info.trace_id).data.spans
    if span.name == "action_response_agent"
]

print(f"Found {len(spans)} action_response_agent spans")

Found 9 action_response_agent spans


[Trace(trace_id=tr-b11d9a98ba12ca9924cb0dd98b5c2661), Trace(trace_id=tr-baf3ea4846cba9ab365b3a2044a99581), Trace(trace_id=tr-b350d8ff770eee4eeb32dc92226cbfd9), Trace(trace_id=tr-935a11230bc23f5cfc7088cc4ea47034), Trace(trace_id=tr-e8850738d939c8b9e3ba8adc63b49645), Trace(trace_id=tr-7a8e97a493805cc88fa1a270263b2b0e), Trace(trace_id=tr-c90c9c1a70a2a22719fe400863e054e4), Trace(trace_id=tr-5245240b660065884970436ad51e79f3), Trace(trace_id=tr-f83ef4a93f7a9247c8df8050d4663afa)]

In [41]:
records = []

for span in spans:
    # Inputs/outputs are stored as JSON strings or dicts
    inputs = span.inputs  # Dict containing student_action, context, etc.
    outputs = span.outputs  # Dict containing DynamicTurnResult fields

    records.append(
        {
            "trace_id": span.trace_id,
            "inputs": inputs,
            "outputs": outputs,
        }
    )

eval_df = pd.DataFrame(records)
eval_df

,trace_id,inputs,outputs
0,tr-b11d9a98ba12ca9924cb0dd98b5c2661,"{'input_data': {'student_action': 'next step',...","{'was_correct': True, 'completion_score': 0.6,..."
1,tr-baf3ea4846cba9ab365b3a2044a99581,{'input_data': {'student_action': 'What do I d...,"{'was_correct': True, 'completion_score': 0.4,..."
2,tr-b350d8ff770eee4eeb32dc92226cbfd9,"{'input_data': {'student_action': 'idk', 'scen...","{'was_correct': True, 'completion_score': 0.2,..."
3,tr-935a11230bc23f5cfc7088cc4ea47034,"{'input_data': {'student_action': 'idk', 'scen...","{'was_correct': True, 'completion_score': 0.2,..."
4,tr-e8850738d939c8b9e3ba8adc63b49645,{'input_data': {'student_action': 'I check the...,"{'was_correct': True, 'completion_score': 0.6,..."
5,tr-7a8e97a493805cc88fa1a270263b2b0e,{'input_data': {'student_action': 'what do I d...,"{'was_correct': True, 'completion_score': 0.4,..."
6,tr-c90c9c1a70a2a22719fe400863e054e4,{'input_data': {'student_action': 'Check the s...,"{'was_correct': True, 'completion_score': 0.2,..."
7,tr-5245240b660065884970436ad51e79f3,"{'input_data': {'student_action': 'Okay, check...","{'was_correct': True, 'completion_score': 0.4,..."
8,tr-f83ef4a93f7a9247c8df8050d4663afa,"{'input_data': {'student_action': 'idk', 'scen...","{'was_correct': True, 'completion_score': 0.2,..."
